In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from scipy import stats

# 1. Define the input files, descriptors, and parameters
DESCRIPTORS = ["BCUT2D_CHGLO", "BertzCT"]
TRAINING_FILE = "experimental_data.csv"
SCREENING_FILE = "aminoacids_21.csv"
TOP_N = 10

# 2. Function to calculate descriptor value from SMILES
def get_descriptor_val(smiles, name):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            calc_func = getattr(Descriptors, name)
            return calc_func(mol)
    except:
        return None
    return None

# 3. Train the models using the experimental data
exp_df = pd.read_csv(TRAINING_FILE)
models = {} # Store slopes and intercepts

print("Training models for 5 descriptors...")
for desc in DESCRIPTORS:
    exp_df[desc] = exp_df['SMILES'].apply(lambda x: get_descriptor_val(x, desc))
    
    mask = exp_df[desc].notnull() & exp_df['Stability_Hours'].notnull()
    slope, intercept, r_val, p_val, _ = stats.linregress(exp_df[mask][desc], exp_df[mask]['Stability_Hours'])
    
    models[desc] = {'slope': slope, 'intercept': intercept, 'r_squared': r_val**2}
    print(f" - {desc}: R² = {r_val**2:.3f}")

# 4. Screen the new dataset and predict stability
screen_df = pd.read_csv(SCREENING_FILE)
name_col = 'Amino_Acid' if 'Amino_Acid' in screen_df.columns else 'SMILES'
top_sets = []

print(f"\nScreening {len(screen_df)} molecules...")

for desc in DESCRIPTORS:
    screen_df[desc] = [get_descriptor_val(s, desc) for s in screen_df['SMILES']]
    
    pred_col = f"Pred_Stability_{desc}"
    screen_df[pred_col] = (models[desc]['slope'] * screen_df[desc]) + models[desc]['intercept']
    
    top_molecules = screen_df.sort_values(by=pred_col, ascending=False).head(TOP_N)[name_col].tolist()
    top_sets.append(set(top_molecules))

# 5. Find common leads across all descriptors and print results
common_leads = set.intersection(*top_sets)

print(f"\nFound {len(common_leads)} leads that appear in the Top {TOP_N} of the 2 lists.")

if common_leads:
    results_mask = screen_df[name_col].isin(common_leads)
    prediction_cols = [f"Pred_Stability_{desc}" for desc in DESCRIPTORS]
    
    chart_df = screen_df[results_mask][[name_col] + prediction_cols]
    
    chart_df['Average_Predicted_Stability'] = chart_df[prediction_cols].mean(axis=1)
    chart_df = chart_df.sort_values(by='Average_Predicted_Stability', ascending=False)
    
    print("-" * 100)
    print("CONSENSUS LEADS PREDICTED STABILITY (Hours):")
    print(chart_df.to_string(index=False))
else:
    print("No overlapping molecules found in the Top 20 for all 5 descriptors.")
    print("Consider increasing TOP_N or checking for low correlation between models.")

Training models for 5 descriptors...
 - BCUT2D_CHGLO: R² = 0.905
 - BertzCT: R² = 0.711

Screening 21 molecules...

Found 6 leads that appear in the Top 10 of the 2 lists.
----------------------------------------------------------------------------------------------------
CONSENSUS LEADS PREDICTED STABILITY (Hours):
Amino_Acid  Pred_Stability_BCUT2D_CHGLO  Pred_Stability_BertzCT  Average_Predicted_Stability
   Glycine                    11.471878                9.473248                    10.472563
 Sarcosine                     7.456877                8.000994                     7.728935
   Alanine                     5.217437                6.945128                     6.081282
    Serine                     2.926695                5.035421                     3.981058
  Cysteine                     2.887261                4.623332                     3.755296
   Proline                     1.805298                1.442170                     1.623734


In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from scipy import stats

# 1. Define the input files, descriptors, and parameters
DESCRIPTORS = ["BCUT2D_CHGLO", "BertzCT"]
TRAINING_FILE = "experimental_data.csv"
SCREENING_FILE = "ccs-lit-167.csv"
TOP_N = 10

# 2. Function to calculate descriptor value from SMILES
def get_descriptor_val(smiles, name):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            calc_func = getattr(Descriptors, name)
            return calc_func(mol)
    except:
        return None
    return None

# 3. Train the models using the experimental data
exp_df = pd.read_csv(TRAINING_FILE)
models = {} # Store slopes and intercepts

print("Training models for 5 descriptors...")
for desc in DESCRIPTORS:
    exp_df[desc] = exp_df['SMILES'].apply(lambda x: get_descriptor_val(x, desc))
    
    mask = exp_df[desc].notnull() & exp_df['Stability_Hours'].notnull()
    slope, intercept, r_val, p_val, _ = stats.linregress(exp_df[mask][desc], exp_df[mask]['Stability_Hours'])
    
    models[desc] = {'slope': slope, 'intercept': intercept, 'r_squared': r_val**2}
    print(f" - {desc}: R² = {r_val**2:.3f}")

# 4. Screen the new dataset and predict stability
screen_df = pd.read_csv(SCREENING_FILE)
name_col = 'Solvent' if 'Solvent' in screen_df.columns else 'SMILES'
top_sets = []

print(f"\nScreening {len(screen_df)} molecules...")

for desc in DESCRIPTORS:
    screen_df[desc] = [get_descriptor_val(s, desc) for s in screen_df['SMILES']]
    
    pred_col = f"Pred_Stability_{desc}"
    screen_df[pred_col] = (models[desc]['slope'] * screen_df[desc]) + models[desc]['intercept']
    
    top_molecules = screen_df.sort_values(by=pred_col, ascending=False).head(TOP_N)[name_col].tolist()
    top_sets.append(set(top_molecules))

# 5. Find common leads across all descriptors and print results
common_leads = set.intersection(*top_sets)

print(f"\nFound {len(common_leads)} leads that appear in the Top {TOP_N} of the 2 lists.")

if common_leads:
    results_mask = screen_df[name_col].isin(common_leads)
    prediction_cols = [f"Pred_Stability_{desc}" for desc in DESCRIPTORS]
    
    chart_df = screen_df[results_mask][[name_col] + prediction_cols]
    
    chart_df['Average_Predicted_Stability'] = chart_df[prediction_cols].mean(axis=1)
    chart_df = chart_df.sort_values(by='Average_Predicted_Stability', ascending=False)
    
    print("-" * 100)
    print("CONSENSUS LEADS PREDICTED STABILITY (Hours):")
    print(chart_df.to_string(index=False))
else:
    print("No overlapping molecules found in the Top 20 for all 5 descriptors.")
    print("Consider increasing TOP_N or checking for low correlation between models.")

Training models for 5 descriptors...
 - BCUT2D_CHGLO: R² = 0.905
 - BertzCT: R² = 0.711

Screening 167 molecules...

Found 7 leads that appear in the Top 10 of the 2 lists.
----------------------------------------------------------------------------------------------------
CONSENSUS LEADS PREDICTED STABILITY (Hours):
           Solvent  Pred_Stability_BCUT2D_CHGLO  Pred_Stability_BertzCT  Average_Predicted_Stability
        Ethylamine                    18.811292               15.480227                    17.145759
       Propylamine                    12.512889               15.107727                    13.810308
ethanolamine (MEA)                    12.162964               14.695638                    13.429301
   ethylenediamine                    11.852573               14.994808                    13.423691
        Butylamine                     8.838593               14.229022                    11.533807
1,3-diaminopropane                     8.518171               14.042772    